In [ ]:
import pandas as pd
import numpy as np
import ast
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
base_path = "/content/drive/MyDrive/archive"

seq_path = f"{base_path}/user_sequences_top1500_user5.csv"
products_path = f"{base_path}/products_top1500.csv"

seq_df = pd.read_csv(seq_path)
products_df = pd.read_csv(products_path)

print(seq_df.shape)
print(products_df.shape)
seq_df.head()

(155461, 2)
(1500, 4)


,user_id,product_sequence
0,1,196 14084 12427 196 12427 13176 196 12427 2513...
1,2,47766 20574 22474 16589 27344 30489 27966 1317...
2,3,9387 16797 39190 47766 21903 39922 24810 38596...
3,7,39275 45066 13249 31683 22963 14332 4920 22963...
4,10,46979 24852 27104 16797 31717 46979 20995 4301...


In [ ]:
def parse_sequence(seq_str):
    return [int(x) for x in str(seq_str).split()]

seq_df["sequence"] = seq_df["product_sequence"].apply(parse_sequence)

seq_df[["user_id", "sequence"]].head()

,user_id,sequence
0,1,"[196, 14084, 12427, 196, 12427, 13176, 196, 12..."
1,2,"[47766, 20574, 22474, 16589, 27344, 30489, 279..."
2,3,"[9387, 16797, 39190, 47766, 21903, 39922, 2481..."
3,7,"[39275, 45066, 13249, 31683, 22963, 14332, 492..."
4,10,"[46979, 24852, 27104, 16797, 31717, 46979, 209..."


In [ ]:
all_products = set()
for seq in seq_df["sequence"]:
    all_products.update(seq)

all_products = sorted(list(all_products))

product_to_idx = {p: i + 1 for i, p in enumerate(all_products)}  # 0은 padding용
idx_to_product = {i + 1: p for i, p in enumerate(all_products)}

num_products = len(product_to_idx)
print("상품 개수:", num_products)

상품 개수: 1500


In [ ]:
seq_df["encoded_sequence"] = seq_df["sequence"].apply(
    lambda seq: [product_to_idx[p] for p in seq if p in product_to_idx]
)

seq_df[["user_id", "encoded_sequence"]].head()

,user_id,encoded_sequence
0,1,"[7, 415, 357, 7, 357, 380, 7, 357, 754, 910, 7..."
1,2,"[1437, 605, 671, 478, 830, 913, 849, 380, 1354..."
2,3,"[270, 483, 1179, 1437, 651, 1199, 742, 1152, 6..."
3,7,"[1181, 1354, 384, 944, 686, 425, 120, 686, 563..."
4,10,"[1415, 747, 819, 483, 945, 1415, 620, 1279, 82..."


In [ ]:
SEQ_LEN = 5

X = []
y = []

for seq in seq_df["encoded_sequence"]:
    if len(seq) <= SEQ_LEN:
        continue
    for i in range(len(seq) - SEQ_LEN):
        X.append(seq[i:i+SEQ_LEN])
        y.append(seq[i+SEQ_LEN])

X = np.array(X, dtype=np.int64)
y = np.array(y, dtype=np.int64)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (18121276, 5)
y shape: (18121276,)


In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = SequenceDataset(X, y)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print("train:", len(train_dataset), "test:", len(test_dataset))

train: 14497020 test: 3624256


In [ ]:
class NextProductLSTM(nn.Module):
    def __init__(self, num_products, embed_dim=64, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(num_products + 1, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_products + 1)

    def forward(self, x):
        x = self.embedding(x)              # (batch, seq_len, embed_dim)
        out, (hidden, cell) = self.lstm(x)
        last_hidden = hidden[-1]           # (batch, hidden_dim)
        logits = self.fc(last_hidden)      # (batch, num_products+1)
        return logits

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

model = NextProductLSTM(num_products=num_products).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

device: cpu


In [ ]:
def topk_accuracy(model, data_loader, k=5):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_x)
            topk = torch.topk(outputs, k=k, dim=1).indices

            for i in range(batch_y.size(0)):
                if batch_y[i] in topk[i]:
                    correct += 1
                total += 1

    return correct / total if total > 0 else 0.0

In [ ]:
EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    acc1 = topk_accuracy(model, test_loader, k=1)
    acc5 = topk_accuracy(model, test_loader, k=5)

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss:.4f} | Top-1 Acc: {acc1:.4f} | Top-5 Acc: {acc5:.4f}")

Epoch 1/10 | Loss: 346310.3498 | Top-1 Acc: 0.0455 | Top-5 Acc: 0.1378
Epoch 2/10 | Loss: 341018.1349 | Top-1 Acc: 0.0467 | Top-5 Acc: 0.1411
Epoch 3/10 | Loss: 339762.5426 | Top-1 Acc: 0.0473 | Top-5 Acc: 0.1425
Epoch 4/10 | Loss: 339081.4527 | Top-1 Acc: 0.0476 | Top-5 Acc: 0.1435
Epoch 5/10 | Loss: 338631.1310 | Top-1 Acc: 0.0478 | Top-5 Acc: 0.1440
Epoch 6/10 | Loss: 338306.4883 | Top-1 Acc: 0.0481 | Top-5 Acc: 0.1447
Epoch 7/10 | Loss: 338054.6800 | Top-1 Acc: 0.0480 | Top-5 Acc: 0.1449
Epoch 8/10 | Loss: 337849.5019 | Top-1 Acc: 0.0485 | Top-5 Acc: 0.1454
Epoch 9/10 | Loss: 337685.7729 | Top-1 Acc: 0.0484 | Top-5 Acc: 0.1454
Epoch 10/10 | Loss: 337549.5474 | Top-1 Acc: 0.0486 | Top-5 Acc: 0.1458


In [ ]:
def recommend_next_products(model, encoded_seq, top_k=5):
    model.eval()

    if len(encoded_seq) < SEQ_LEN:
        return []

    input_seq = encoded_seq[-SEQ_LEN:]
    input_tensor = torch.tensor([input_seq], dtype=torch.long).to(device)

    with torch.no_grad():
        outputs = model(input_tensor)
        topk_indices = torch.topk(outputs, k=top_k, dim=1).indices[0].tolist()

    return topk_indices

In [ ]:
def decode_products(pred_indices):
    product_ids = [idx_to_product[idx] for idx in pred_indices if idx in idx_to_product]
    result = products_df[products_df["product_id"].isin(product_ids)][["product_id", "product_name", "aisle_id", "department_id"]]
    return result

sample_user_id = seq_df.iloc[0]["user_id"]
sample_seq = seq_df.iloc[0]["encoded_sequence"]

pred_indices = recommend_next_products(model, sample_seq, top_k=5)
result_df = decode_products(pred_indices)

print("sample_user_id:", sample_user_id)
print("pred_indices:", pred_indices)
result_df

sample_user_id: 1
pred_indices: [1166, 1194, 7, 754, 1126]


,product_id,product_name,aisle_id,department_id
6,196,Soda,77,7
753,25133,Organic String Cheese,21,16
1125,37710,Trail Mix,125,19
1165,38928,0% Greek Strained Yogurt,120,16
1193,39657,Milk Chocolate Almonds,45,19


In [ ]:
recommendation_rows = []

for _, row in seq_df.iterrows():
    user_id = row["user_id"]
    encoded_seq = row["encoded_sequence"]

    pred_indices = recommend_next_products(model, encoded_seq, top_k=5)

    for rank, pred_idx in enumerate(pred_indices, start=1):
        product_id = idx_to_product.get(pred_idx)
        if product_id is not None:
            recommendation_rows.append({
                "user_id": user_id,
                "product_id": product_id,
                "rank": rank,
                "model_name": "lstm_v1"
            })

recommendations_df = pd.DataFrame(recommendation_rows)
recommendations_df.to_csv(f"{base_path}/recommendations_top5_lstm.csv", index=False)

print(recommendations_df.shape)
recommendations_df.head()